<a href="https://colab.research.google.com/github/AntonDozhdikov/AntonDozhdikov/blob/main/%D0%AD%D0%BA%D1%81%D0%BF%D0%B5%D1%80%D0%B8%D0%BC%D0%B5%D0%BD%D1%82_2.1.%D1%81%D0%B8%D0%BD%D1%82%D0%B5%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5_%D1%80%D0%B5%D1%81%D0%BF%D0%BE%D0%BD%D0%B4%D0%B5%D0%BD%D1%82%D1%8B1200.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# ЭКСПЕРИМЕНТ 2: Синтетические респонденты или избиратели-синтетики?
# Модель: ai-forever/ru-en-RoSBERTa (ruRoBERTa-large, 404M параметров, 1024-мерные эмбеддинги)
# MARL: 5 кооперирующих агентов
# Среда: Google Colab, GPU T4
# Время выполнения: ~30–60 минут
# ============================================================

# ── БЛОК 0: Установка зависимостей ──────────────────────────
# Раскомментируй и запусти один раз:
!pip install transformers torch scipy seaborn matplotlib pandas numpy --quiet

In [2]:
# ── БЛОК 1: Импорты ─────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, kruskal, wilcoxon
from scipy.spatial.distance import jensenshannon
import torch
from transformers import AutoTokenizer, AutoModel
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 65)
print("ЭКСПЕРИМЕНТ 2: Domain Transfer в LLM-панелях")
print("Модель: ai-forever/ru-en-RoSBERTa (ruRoBERTa-large, 404M)")
print("MARL: 5 кооперирующих агентов")
print("=" * 65)

ЭКСПЕРИМЕНТ 2: Domain Transfer в LLM-панелях
Модель: ai-forever/ru-en-RoSBERTa (ruRoBERTa-large, 404M)
MARL: 5 кооперирующих агентов


In [3]:
# ── БЛОК 2: Константы ───────────────────────────────────────

SEEDS        = list(range(42, 72))   # 30 seeds — те же, что в Exp.1
N_PERSONAS   = 1200
N_QUESTIONS  = 20
N_AGENTS     = 5                     # ← увеличено с 3 до 5

# Те же вопросы, что в Эксперименте 1 (воспроизводимость)

MARKETING_QUESTIONS = [
    "Купили бы вы смартфон за 30 000 рублей?",
    "Как часто вы заказываете еду с доставкой?",
    "Оцените вашу лояльность к основному банку (1–5)",
    "Готовы ли вы переплатить 20% за органические продукты?",
    "Насколько важен бренд при выборе одежды?",
    "Пользуетесь ли вы подпиской на стриминговые сервисы?",
    "Как часто вы совершаете покупки онлайн?",
    "Оцените удовлетворённость качеством интернета дома (1–5)",
    "Готовы ли вы платить за премиальную доставку (1 день)?",
    "Насколько важны для вас скидки и акции при покупке?",
    "Используете ли вы кешбэк-программы банков?",
    "Оцените важность экологичности упаковки товара (1–5)",
    "Как часто вы посещаете торговые центры?",
    "Пользуетесь ли вы маркетплейсами (Ozon, Wildberries)?",
    "Оцените, насколько реклама влияет на ваш выбор товара (1–5)",
    "Готовы ли вы сменить банк ради лучших условий?",
    "Насколько важна скорость обслуживания в кафе?",
    "Оцените уровень доверия к онлайн-отзывам товаров (1–5)",
    "Как часто вы обновляете смартфон?",
    "Готовы ли вы платить за персонализированные рекомендации?"
]

POLITICAL_QUESTIONS = [
    "За какую партию вы проголосовали бы на ближайших выборах?",
    "Как вы оцениваете деятельность президента? (1–5)",
    "Насколько вы доверяете государственным институтам?",
    "Считаете ли вы выборы в России честными?",
    "Насколько вы доверяете СМИ при освещении политики?",
    "Поддерживаете ли вы текущий курс внешней политики?",
    "Оцените уровень коррупции в органах власти (1–5)",
    "Считаете ли вы оппозицию реальной политической силой?",
    "Насколько важна для вас свобода слова в интернете?",
    "Поддерживаете ли вы увеличение государственных расходов на оборону?",
    "Доверяете ли вы результатам социологических опросов ВЦИОМ?",
    "Оцените справедливость распределения доходов в стране (1–5)",
    "Участвовали бы вы в легальном политическом митинге?",
    "Считаете ли вы необходимым развитие гражданского общества?",
    "Насколько вы доверяете судебной системе?",
    "Поддерживаете ли вы санкции против других стран?",
    "Оцените экономическую политику правительства (1–5)",
    "Считаете ли вы необходимым изменение Конституции?",
    "Насколько вы идентифицируете себя с интересами государства?",
    "Поддерживаете ли вы снижение пенсионного возраста?"
]

assert len(MARKETING_QUESTIONS) == N_QUESTIONS
assert len(POLITICAL_QUESTIONS) == N_QUESTIONS
print(f"Вопросов D_M: {len(MARKETING_QUESTIONS)}, D_P: {len(POLITICAL_QUESTIONS)}")


Вопросов D_M: 20, D_P: 20


In [4]:
# ── БЛОК 3: Загрузка модели ──────────────────────────────────

print("\nЗагружаю модель ai-forever/ru-en-RoSBERTa ...")
print("(ruRoBERTa-large, 404M параметров, ~800 MB, публичная)")
t0 = time.time()

# Основная модель — ai-forever/ru-en-RoSBERTa (руRoBERTa-large, 404M, публичная)
# Запасная — DeepPavlov/rubert-base-cased (ruBERT, 180M, публичная, 768D)
MODEL_NAME = 'ai-forever/ru-en-RoSBERTa'
try:
    tokenizer2 = AutoTokenizer.from_pretrained(MODEL_NAME)
    model2 = AutoModel.from_pretrained(MODEL_NAME)
    EMB_DIM_ACTUAL = 1024
    print(f"  Загружена основная модель: {MODEL_NAME}")
except Exception as e:
    print(f"  Основная модель недоступна ({e})")
    print("  Переключаюсь на запасную: DeepPavlov/rubert-base-cased")
    MODEL_NAME = 'DeepPavlov/rubert-base-cased'
    tokenizer2 = AutoTokenizer.from_pretrained(MODEL_NAME)
    model2 = AutoModel.from_pretrained(MODEL_NAME)
    EMB_DIM_ACTUAL = 768
    print(f"  Загружена запасная модель: {MODEL_NAME}")
model2.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model2 = model2.to(device)

elapsed = time.time() - t0
n_params = sum(p.numel() for p in model2.parameters())
EMB_DIM = EMB_DIM_ACTUAL  # 1024 для ru-en-RoSBERTa, 768 для DeepPavlov/rubert-base-cased

print(f"Модель загружена за {elapsed:.1f}s. Устройство: {device.upper()}")
print(f"Параметров: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"Размерность эмбеддингов: {EMB_DIM} (ruRoBERTa-large)")


Загружаю модель ai-forever/ru-en-RoSBERTa ...
(ruRoBERTa-large, 404M параметров, ~800 MB, публичная)


config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.49M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ai-forever/ru-en-RoSBERTa
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Загружена основная модель: ai-forever/ru-en-RoSBERTa
Модель загружена за 20.6s. Устройство: CUDA
Параметров: 404,757,504 (404.8M)
Размерность эмбеддингов: 1024 (ruRoBERTa-large)


In [5]:
# ── БЛОК 4: Вспомогательные функции ─────────────────────────

def get_embedding_v2(text: str) -> np.ndarray:
    """Эмбеддинг через ru-en-RoSBERTa (CLS pooling, рекомендован авторами)."""
    inputs = tokenizer2(
        text, return_tensors='pt',
        truncation=True, max_length=128, padding=True
    ).to(device)
    with torch.no_grad():
        output = model2(**inputs)
    # CLS pooling — рекомендован авторами ru-en-RoSBERTa
    emb = output.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return emb / (np.linalg.norm(emb) + 1e-9)


def cosine_sim_v2(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def simulate_response_v2(q_emb: np.ndarray, p_emb: np.ndarray,
                          domain: str, rng: np.random.Generator) -> float:
    """
    Симуляция ответа с доменным штрафом.
    Для Exp.2 используем те же σ, что в Exp.1 — для сравнимости.
    σ_M = 0.15 (in-domain), σ_P = 0.375 (out-of-domain, ×2.5)
    """
    noise_std = 0.15 if domain == 'marketing' else 0.375
    base = cosine_sim_v2(q_emb, p_emb)
    return float(np.clip(base + rng.normal(0, noise_std), 0.0, 1.0))


def responses_to_dist(responses: list, n_bins: int = 10) -> np.ndarray:
    hist, _ = np.histogram(responses, bins=n_bins, range=(0, 1))
    hist = hist.astype(float) + 1e-9
    return hist / hist.sum()


def compute_jsd_v2(q_emb: np.ndarray, persona_embs: list,
                   domain: str, rng: np.random.Generator) -> float:
    synth = [simulate_response_v2(q_emb, p, domain, rng) for p in persona_embs]
    synth_dist = responses_to_dist(synth)
    ref_dist = np.ones(len(synth_dist)) / len(synth_dist)
    return float(jensenshannon(synth_dist, ref_dist))


print("Вспомогательные функции определены.")


Вспомогательные функции определены.


In [6]:
# ── БЛОК 5: Эмбеддинги вопросов (один раз) ──────────────────

print("\nВычисляю эмбеддинги вопросов через ruBERT-base-cased ...")
t0 = time.time()
mkt_embs2 = [get_embedding_v2(q) for q in MARKETING_QUESTIONS]
pol_embs2  = [get_embedding_v2(q) for q in POLITICAL_QUESTIONS]
print(f"Эмбеддинги вычислены за {time.time()-t0:.1f}s. Размер: {mkt_embs2[0].shape[0]}D")



Вычисляю эмбеддинги вопросов через ruBERT-base-cased ...
Эмбеддинги вычислены за 2.8s. Размер: 1024D


In [7]:
# ── БЛОК 6: Основной цикл — 30 seeds ────────────────────────

print("\nОсновной цикл (30 seeds × 1200 персон × 20 вопросов) ...")
print("Ожидаемое время на GPU T4: ~3–8 минут\n")

results2 = []
t0 = time.time()

for i, seed in enumerate(SEEDS):
    rng = np.random.default_rng(seed)

    # Персоны в пространстве 768D (полный ruBERT)
    persona_embs = [rng.standard_normal(EMB_DIM) for _ in range(N_PERSONAS)]
    persona_embs = [p / (np.linalg.norm(p) + 1e-9) for p in persona_embs]

    jsd_m_list = [compute_jsd_v2(q, persona_embs, 'marketing', rng)
                  for q in mkt_embs2]
    jsd_p_list = [compute_jsd_v2(q, persona_embs, 'political', rng)
                  for q in pol_embs2]

    results2.append({
        'seed':      seed,
        'jsd_m':     float(np.mean(jsd_m_list)),
        'jsd_p':     float(np.mean(jsd_p_list)),
        'jsd_m_std': float(np.std(jsd_m_list)),
        'jsd_p_std': float(np.std(jsd_p_list)),
    })

    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f"  Seeds выполнено: {i+1}/30  [{elapsed:.1f}s]")

df2 = pd.DataFrame(results2)
print(f"\nОсновной цикл завершён за {time.time()-t0:.1f}s.")



Основной цикл (30 seeds × 1200 персон × 20 вопросов) ...
Ожидаемое время на GPU T4: ~3–8 минут

  Seeds выполнено: 10/30  [20.4s]
  Seeds выполнено: 20/30  [35.0s]
  Seeds выполнено: 30/30  [45.9s]

Основной цикл завершён за 45.9s.


In [8]:
# ── БЛОК 7: MARL-симуляция (5 агентов) ──────────────────────

print(f"\nMARL-симуляция ({N_AGENTS} агентов, кооперативный режим) ...")

# 5 агентов с симметричными смещениями
agent_offsets_5 = [-0.10, -0.05, 0.0, +0.05, +0.10]

marl_results2 = []
t0 = time.time()

for seed in SEEDS:
    rng = np.random.default_rng(seed + 2000)   # отдельный генератор

    persona_embs = [rng.standard_normal(EMB_DIM) for _ in range(N_PERSONAS)]
    persona_embs = [p / (np.linalg.norm(p) + 1e-9) for p in persona_embs]

    marl_jsd_m_list = []
    marl_jsd_p_list = []

    for q_emb in mkt_embs2:
        agent_responses = []
        for offset in agent_offsets_5:
            resp = [np.clip(simulate_response_v2(q_emb, p, 'marketing', rng) + offset, 0, 1)
                    for p in persona_embs]
            agent_responses.append(resp)
        consensus = list(np.mean(agent_responses, axis=0))
        synth_dist = responses_to_dist(consensus)
        ref_dist = np.ones(len(synth_dist)) / len(synth_dist)
        marl_jsd_m_list.append(float(jensenshannon(synth_dist, ref_dist)))

    for q_emb in pol_embs2:
        agent_responses = []
        for offset in agent_offsets_5:
            resp = [np.clip(simulate_response_v2(q_emb, p, 'political', rng) + offset, 0, 1)
                    for p in persona_embs]
            agent_responses.append(resp)
        consensus = list(np.mean(agent_responses, axis=0))
        synth_dist = responses_to_dist(consensus)
        ref_dist = np.ones(len(synth_dist)) / len(synth_dist)
        marl_jsd_p_list.append(float(jensenshannon(synth_dist, ref_dist)))

    marl_results2.append({
        'seed':       seed,
        'marl_jsd_m': float(np.mean(marl_jsd_m_list)),
        'marl_jsd_p': float(np.mean(marl_jsd_p_list)),
    })

df_marl2 = pd.DataFrame(marl_results2)
print(f"MARL-симуляция завершена за {time.time()-t0:.1f}s.")


MARL-симуляция (5 агентов, кооперативный режим) ...
MARL-симуляция завершена за 192.8s.


In [9]:
# ── БЛОК 8: Статистические тесты ────────────────────────────

print("\n" + "=" * 65)
print("СТАТИСТИЧЕСКИЕ ТЕСТЫ — Эксперимент 2")
print("=" * 65)

# Тест Манна–Уитни (H₁: JSD_P > JSD_M)
stat_mw2, p_mw2 = mannwhitneyu(df2['jsd_p'], df2['jsd_m'], alternative='greater')
n = len(df2)
mu_u  = n * n / 2
sig_u = np.sqrt(n * n * (2 * n + 1) / 12)
z_mw2 = (stat_mw2 - mu_u) / sig_u
r_mw2 = abs(z_mw2) / np.sqrt(2 * n)

print(f"\nТест Манна–Уитни (H₁: JSD_P > JSD_M):")
print(f"  U  = {stat_mw2:.2f}")
print(f"  Z  = {z_mw2:.4f}")
print(f"  p  = {p_mw2:.8f}")
print(f"  r  = {r_mw2:.4f}")

# Bootstrap CI
rng_boot2 = np.random.default_rng(999)
N_BOOT = 10_000

def bootstrap_ci2(data, n_boot=N_BOOT, ci=0.95):
    boot = [rng_boot2.choice(data, size=len(data), replace=True).mean()
            for _ in range(n_boot)]
    return np.percentile(boot, [(1-ci)/2*100, (1+ci)/2*100])

ci_m2 = bootstrap_ci2(df2['jsd_m'].values)
ci_p2 = bootstrap_ci2(df2['jsd_p'].values)

print(f"\nBootstrap 95% CI (10 000 репликаций):")
print(f"  JSD_маркетинг: M = {df2['jsd_m'].mean():.4f} ± {df2['jsd_m'].std():.4f}  "
      f"[{ci_m2[0]:.4f}, {ci_m2[1]:.4f}]")
print(f"  JSD_политика:  M = {df2['jsd_p'].mean():.4f} ± {df2['jsd_p'].std():.4f}  "
      f"[{ci_p2[0]:.4f}, {ci_p2[1]:.4f}]")

delta2 = df2['jsd_p'].mean() - df2['jsd_m'].mean()
delta_pct2 = delta2 / df2['jsd_m'].mean() * 100
print(f"\n  Δ (абс.) = {delta2:.4f}  ({delta_pct2:.1f}%)")

# Kruskal–Wallis
g1_2 = df2[df2['seed'] < 52]['jsd_p'].values
g2_2 = df2[(df2['seed'] >= 52) & (df2['seed'] < 62)]['jsd_p'].values
g3_2 = df2[df2['seed'] >= 62]['jsd_p'].values
stat_kw2, p_kw2 = kruskal(g1_2, g2_2, g3_2)
print(f"\nКраскел–Уоллис: H = {stat_kw2:.4f},  p = {p_kw2:.4f}")

# MARL (5 агентов) vs одиночный агент
marl_delta_p2 = df_marl2['marl_jsd_p'].mean() - df2['jsd_p'].mean()
marl_delta_m2 = df_marl2['marl_jsd_m'].mean() - df2['jsd_m'].mean()
stat_marl2, p_marl2 = mannwhitneyu(df_marl2['marl_jsd_p'], df2['jsd_p'],
                                    alternative='greater')
print(f"\nMARL (5 агентов) vs Одиночный (политика):")
print(f"  JSD_одиночный = {df2['jsd_p'].mean():.4f}")
print(f"  JSD_MARL-5    = {df_marl2['marl_jsd_p'].mean():.4f}")
print(f"  Δ_MARL_P      = {marl_delta_p2:+.4f}")
print(f"  U = {stat_marl2:.2f}, p = {p_marl2:.2e}")
print(f"\nMARL (5 агентов) vs Одиночный (маркетинг):")
print(f"  JSD_одиночный = {df2['jsd_m'].mean():.4f}")
print(f"  JSD_MARL-5    = {df_marl2['marl_jsd_m'].mean():.4f}")
print(f"  Δ_MARL_M      = {marl_delta_m2:+.4f}")

# ── Сравнение Exp.1 vs Exp.2 (Wilcoxon для парных данных) ───
# Данные Exp.1 (из результатов эксперимента 1)
jsd_m_exp1 = np.array([0.6030] * 30)  # приближение, используем mean
jsd_p_exp1 = np.array([0.4696] * 30)

# Реальные данные из Exp.2 vs статистика Exp.1 (попарное сравнение по seeds)
# Используем данные Exp.2 для структурного сравнения
delta_exp2 = df2['jsd_m'].values - df2['jsd_p'].values   # разность M-P по seeds

print(f"\n── Сравнение Exp.1 vs Exp.2 ──")
print(f"Exp.1: Δ(JSD_M - JSD_P) = {0.6030 - 0.4696:.4f}  (по агрегированным данным)")
print(f"Exp.2: Δ(JSD_M - JSD_P) = {delta2*-1:.4f}  (среднее по {n} seeds)")
print(f"Exp.2 / Exp.1 ratio по Δ: {abs(delta2) / abs(0.6030 - 0.4696):.3f}")

# Тест Манна–Уитни для сравнения абс.разниц между моделями
# (для включения в статью — effect size разницы между Exp.1 и Exp.2)
stat_cross, p_cross = mannwhitneyu(
    df2['jsd_m'].values - df2['jsd_p'].values,
    [0.6030 - 0.4696] * 30,   # константа Exp.1
    alternative='two-sided'
)
print(f"Mann–Whitney (Exp.2 Δ vs Exp.1 Δ): U = {stat_cross:.2f}, p = {p_cross:.4f}")



СТАТИСТИЧЕСКИЕ ТЕСТЫ — Эксперимент 2

Тест Манна–Уитни (H₁: JSD_P > JSD_M):
  U  = 0.00
  Z  = -6.6530
  p  = 1.00000000
  r  = 0.8589

Bootstrap 95% CI (10 000 репликаций):
  JSD_маркетинг: M = 0.5978 ± 0.0011  [0.5974, 0.5981]
  JSD_политика:  M = 0.4360 ± 0.0020  [0.4353, 0.4367]

  Δ (абс.) = -0.1618  (-27.1%)

Краскел–Уоллис: H = 4.5316,  p = 0.1037

MARL (5 агентов) vs Одиночный (политика):
  JSD_одиночный = 0.4360
  JSD_MARL-5    = 0.5206
  Δ_MARL_P      = +0.0846
  U = 900.00, p = 1.51e-11

MARL (5 агентов) vs Одиночный (маркетинг):
  JSD_одиночный = 0.5978
  JSD_MARL-5    = 0.6499
  Δ_MARL_M      = +0.0522

── Сравнение Exp.1 vs Exp.2 ──
Exp.1: Δ(JSD_M - JSD_P) = 0.1334  (по агрегированным данным)
Exp.2: Δ(JSD_M - JSD_P) = 0.1618  (среднее по 30 seeds)
Exp.2 / Exp.1 ratio по Δ: 1.213
Mann–Whitney (Exp.2 Δ vs Exp.1 Δ): U = 900.00, p = 0.0000


In [10]:
# ── БЛОК 9: Сводная таблица Exp.2 ───────────────────────────

print("\n" + "=" * 65)
print("ТАБЛИЦА 2 — Сводные результаты Эксперимента 2 (ru-en-RoSBERTa)")
print("=" * 65)

table2 = pd.DataFrame({
    'Показатель': [
        'JSD (M ± SD)',
        '95% Bootstrap CI',
        'MARL-5 JSD (M ± SD)',
        'Mann–Whitney U',
        'p-value',
        'Effect size r',
        'N (seeds)',
    ],
    'D_M (маркетинг)': [
        f"{df2['jsd_m'].mean():.4f} ± {df2['jsd_m'].std():.4f}",
        f"[{ci_m2[0]:.4f}; {ci_m2[1]:.4f}]",
        f"{df_marl2['marl_jsd_m'].mean():.4f} ± {df_marl2['marl_jsd_m'].std():.4f}",
        '—', '—', '—', str(n),
    ],
    'D_P (политика)': [
        f"{df2['jsd_p'].mean():.4f} ± {df2['jsd_p'].std():.4f}",
        f"[{ci_p2[0]:.4f}; {ci_p2[1]:.4f}]",
        f"{df_marl2['marl_jsd_p'].mean():.4f} ± {df_marl2['marl_jsd_p'].std():.4f}",
        f"{stat_mw2:.1f}",
        f"{p_mw2:.2e}",
        f"{r_mw2:.4f}",
        str(n),
    ],
})
print(table2.to_string(index=False))



ТАБЛИЦА 2 — Сводные результаты Эксперимента 2 (ru-en-RoSBERTa)
         Показатель  D_M (маркетинг)   D_P (политика)
       JSD (M ± SD)  0.5978 ± 0.0011  0.4360 ± 0.0020
   95% Bootstrap CI [0.5974; 0.5981] [0.4353; 0.4367]
MARL-5 JSD (M ± SD)  0.6499 ± 0.0007  0.5206 ± 0.0012
     Mann–Whitney U                —              0.0
            p-value                —         1.00e+00
      Effect size r                —           0.8589
          N (seeds)               30               30


In [11]:
# ── БЛОК 9: Сводная таблица Exp.2 ───────────────────────────

print("\n" + "=" * 65)
print("ТАБЛИЦА 2 — Сводные результаты Эксперимента 2 (ru-en-RoSBERTa)")
print("=" * 65)

table2 = pd.DataFrame({
    'Показатель': [
        'JSD (M ± SD)',
        '95% Bootstrap CI',
        'MARL-5 JSD (M ± SD)',
        'Mann–Whitney U',
        'p-value',
        'Effect size r',
        'N (seeds)',
    ],
    'D_M (маркетинг)': [
        f"{df2['jsd_m'].mean():.4f} ± {df2['jsd_m'].std():.4f}",
        f"[{ci_m2[0]:.4f}; {ci_m2[1]:.4f}]",
        f"{df_marl2['marl_jsd_m'].mean():.4f} ± {df_marl2['marl_jsd_m'].std():.4f}",
        '—', '—', '—', str(n),
    ],
    'D_P (политика)': [
        f"{df2['jsd_p'].mean():.4f} ± {df2['jsd_p'].std():.4f}",
        f"[{ci_p2[0]:.4f}; {ci_p2[1]:.4f}]",
        f"{df_marl2['marl_jsd_p'].mean():.4f} ± {df_marl2['marl_jsd_p'].std():.4f}",
        f"{stat_mw2:.1f}",
        f"{p_mw2:.2e}",
        f"{r_mw2:.4f}",
        str(n),
    ],
})
print(table2.to_string(index=False))



ТАБЛИЦА 2 — Сводные результаты Эксперимента 2 (ru-en-RoSBERTa)
         Показатель  D_M (маркетинг)   D_P (политика)
       JSD (M ± SD)  0.5978 ± 0.0011  0.4360 ± 0.0020
   95% Bootstrap CI [0.5974; 0.5981] [0.4353; 0.4367]
MARL-5 JSD (M ± SD)  0.6499 ± 0.0007  0.5206 ± 0.0012
     Mann–Whitney U                —              0.0
            p-value                —         1.00e+00
      Effect size r                —           0.8589
          N (seeds)               30               30


In [12]:
# ── БЛОК 11: Визуализация Exp.2 (3 панели) ──────────────────

print("\nСтрою Рисунок 2 (Exp.2, 3 панели)...")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    'Рисунок 2. JSD синтетических ответов — Эксперимент 2\n'
    '(ru-en-RoSBERTa / ruRoBERTa-large, 5-агентный MARL, 30 seeds)',
    fontsize=11, fontweight='bold', y=1.02
)

palette = {'D_M (маркетинг)': '#4878CF', 'D_P (политика)': '#D65F5F'}

# Панель A: Boxplot
plot_df2 = pd.DataFrame({
    'JSD': list(df2['jsd_m']) + list(df2['jsd_p']),
    'Домен': ['D_M (маркетинг)'] * len(df2) + ['D_P (политика)'] * len(df2)
})
sns.boxplot(data=plot_df2, x='Домен', y='JSD', palette=palette, ax=axes[0],
            width=0.5, flierprops=dict(marker='o', markersize=4))
axes[0].set_title('A. Box plot JSD (Exp.2)\n(30 seeds, ru-en-RoSBERTa)', fontsize=10)
axes[0].set_xlabel('')
axes[0].set_ylabel('JSD')
y_max = plot_df2['JSD'].max() + 0.015
axes[0].annotate(
    f'p = {p_mw2:.2e}\nr = {r_mw2:.3f}',
    xy=(0.5, y_max), xycoords=('axes fraction', 'data'),
    ha='center', fontsize=9, color='darkgreen',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray')
)

# Панель B: Scatter по seeds
sc = axes[1].scatter(df2['jsd_m'], df2['jsd_p'],
                     c=df2['seed'], cmap='viridis', s=60, alpha=0.85, zorder=3)
lims = [min(df2['jsd_m'].min(), df2['jsd_p'].min()) - 0.01,
        max(df2['jsd_m'].max(), df2['jsd_p'].max()) + 0.01]
axes[1].plot(lims, lims, 'k--', linewidth=1, alpha=0.5, label='JSD_P = JSD_M')
axes[1].set_title('B. JSD_M vs JSD_P (Exp.2)\n(seeds, цвет = seed)', fontsize=10)
axes[1].set_xlabel('JSD D_M (маркетинг)')
axes[1].set_ylabel('JSD D_P (политика)')
axes[1].legend(fontsize=8)
axes[1].set_xlim(lims); axes[1].set_ylim(lims)

# Панель C: Сравнение Exp.1 vs Exp.2 (4 группы: одиночный D_M, одиночный D_P × 2 эксперимента)
categories = ['D_M\nExp.1', 'D_M\nExp.2', 'D_P\nExp.1', 'D_P\nExp.2']
values_compare = [0.6030, df2['jsd_m'].mean(), 0.4696, df2['jsd_p'].mean()]
colors_compare = ['#6B8DCF', '#4878CF', '#E08080', '#D65F5F']
bars2 = axes[2].bar(categories, values_compare, color=colors_compare,
                    width=0.55, edgecolor='white')
axes[2].set_title('C. Сравнение Exp.1 vs Exp.2\n(средний JSD, одиночный агент)', fontsize=10)
axes[2].set_ylabel('Средний JSD')
for bar, val in zip(bars2, values_compare):
    axes[2].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8.5)

plt.tight_layout()
FIGURE2_PATH = 'figure2_experiment2_rubert_base.png'
plt.savefig(FIGURE2_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f"Рисунок 2 сохранён: {FIGURE2_PATH}")



Строю Рисунок 2 (Exp.2, 3 панели)...
Рисунок 2 сохранён: figure2_experiment2_rubert_base.png


In [13]:
# ── БЛОК 12: МАRL-сравнительный рисунок (Exp.1 vs Exp.2) ────

print("Строю Рисунок 3 (MARL: Exp.1 3-агентный vs Exp.2 5-агентный)...")

fig3, axes3 = plt.subplots(1, 2, figsize=(12, 5))
fig3.suptitle(
    'Рисунок 3. Эффект MARL-кооперации: 3 агента (Exp.1) vs 5 агентов (Exp.2)',
    fontsize=11, fontweight='bold', y=1.02
)

# Панель A: Boxplot разниц JSD_M - JSD_P по seeds для Exp.2
delta2_series = df2['jsd_m'] - df2['jsd_p']
axes3[0].hist(delta2_series, bins=12, color='#4878CF', edgecolor='white', alpha=0.8)
axes3[0].axvline(delta2_series.mean(), color='red', linestyle='--',
                  linewidth=1.5, label=f'Mean = {delta2_series.mean():.4f}')
axes3[0].axvline(0.1334, color='green', linestyle=':', linewidth=1.5,
                  label='Exp.1 Δ = 0.1334')
axes3[0].set_title('A. Распределение Δ(JSD_M − JSD_P)\nExp.2 по 30 seeds', fontsize=10)
axes3[0].set_xlabel('Δ JSD')
axes3[0].set_ylabel('Частота')
axes3[0].legend(fontsize=8)

# Панель B: 4 столбца — MARL эффект в обоих экспериментах
cat_marl = ['D_M\n3 агента\n(Exp.1)', 'D_M\n5 агентов\n(Exp.2)',
            'D_P\n3 агента\n(Exp.1)', 'D_P\n5 агентов\n(Exp.2)']
val_marl = [0.6378, df_marl2['marl_jsd_m'].mean(),
            0.5039, df_marl2['marl_jsd_p'].mean()]
cols_marl = ['#5A7FBF', '#4878CF', '#C05050', '#D65F5F']
bars3 = axes3[1].bar(cat_marl, val_marl, color=cols_marl, width=0.55, edgecolor='white')
axes3[1].set_title('B. MARL JSD: 3 vs 5 агентов\n(оба эксперимента)', fontsize=10)
axes3[1].set_ylabel('Средний JSD (MARL)')
for bar, val in zip(bars3, val_marl):
    axes3[1].text(bar.get_x() + bar.get_width() / 2,
                  bar.get_height() + 0.003,
                  f'{val:.4f}', ha='center', va='bottom', fontsize=8.5)

plt.tight_layout()
FIGURE3_PATH = 'figure3_marl_comparison.png'
plt.savefig(FIGURE3_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f"Рисунок 3 сохранён: {FIGURE3_PATH}")

Строю Рисунок 3 (MARL: Exp.1 3-агентный vs Exp.2 5-агентный)...
Рисунок 3 сохранён: figure3_marl_comparison.png


In [14]:
# ── БЛОК 13: ИТОГОВЫЙ ВЫВОД ──────────────────────

print("\n" + "=" * 65)
print("ИТОГОВЫЕ ЧИСЛА EXP.2 ")
print("=" * 65)

print(f"""
[EXP2_MODEL]              = ai-forever/ru-en-RoSBERTa
[EXP2_N_PARAMS]           = {n_params:,}
[EXP2_EMB_DIM]            = {EMB_DIM}
[EXP2_N_AGENTS]           = {N_AGENTS}

[EXP2_JSD_M_MEAN]         = {df2['jsd_m'].mean():.4f}
[EXP2_JSD_M_STD]          = {df2['jsd_m'].std():.4f}
[EXP2_JSD_M_CI_LO]        = {ci_m2[0]:.4f}
[EXP2_JSD_M_CI_HI]        = {ci_m2[1]:.4f}

[EXP2_JSD_P_MEAN]         = {df2['jsd_p'].mean():.4f}
[EXP2_JSD_P_STD]          = {df2['jsd_p'].std():.4f}
[EXP2_JSD_P_CI_LO]        = {ci_p2[0]:.4f}
[EXP2_JSD_P_CI_HI]        = {ci_p2[1]:.4f}

[EXP2_DELTA_ABS]          = {delta2:.4f}
[EXP2_DELTA_PCT]          = {delta_pct2:.1f}

[EXP2_MW_U]               = {stat_mw2:.2f}
[EXP2_MW_Z]               = {z_mw2:.4f}
[EXP2_MW_P]               = {p_mw2:.2e}
[EXP2_EFFECT_R]           = {r_mw2:.4f}

[EXP2_KRUSKAL_H]          = {stat_kw2:.4f}
[EXP2_KRUSKAL_P]          = {p_kw2:.4f}

[EXP2_MARL_JSD_M]         = {df_marl2['marl_jsd_m'].mean():.4f}
[EXP2_MARL_JSD_P]         = {df_marl2['marl_jsd_p'].mean():.4f}
[EXP2_MARL_DELTA_P]       = {marl_delta_p2:+.4f}
[EXP2_MARL_DELTA_M]       = {marl_delta_m2:+.4f}
[EXP2_MARL_U]             = {stat_marl2:.2f}
[EXP2_MARL_P]             = {p_marl2:.2e}

[EXP2_CROSS_U]            = {stat_cross:.2f}
[EXP2_CROSS_P]            = {p_cross:.4f}
[EXP2_DELTA_RATIO]        = {abs(delta2) / abs(0.6030 - 0.4696):.3f}
""")

print("=" * 65)
print("Эксперимент 2 завершён.")
print(f"Сохранены файлы: {FIGURE2_PATH}, {FIGURE3_PATH}")
print("=" * 65)


ИТОГОВЫЕ ЧИСЛА EXP.2 

[EXP2_MODEL]              = ai-forever/ru-en-RoSBERTa
[EXP2_N_PARAMS]           = 404,757,504
[EXP2_EMB_DIM]            = 1024
[EXP2_N_AGENTS]           = 5

[EXP2_JSD_M_MEAN]         = 0.5978
[EXP2_JSD_M_STD]          = 0.0011
[EXP2_JSD_M_CI_LO]        = 0.5974
[EXP2_JSD_M_CI_HI]        = 0.5981

[EXP2_JSD_P_MEAN]         = 0.4360
[EXP2_JSD_P_STD]          = 0.0020
[EXP2_JSD_P_CI_LO]        = 0.4353
[EXP2_JSD_P_CI_HI]        = 0.4367

[EXP2_DELTA_ABS]          = -0.1618
[EXP2_DELTA_PCT]          = -27.1

[EXP2_MW_U]               = 0.00
[EXP2_MW_Z]               = -6.6530
[EXP2_MW_P]               = 1.00e+00
[EXP2_EFFECT_R]           = 0.8589

[EXP2_KRUSKAL_H]          = 4.5316
[EXP2_KRUSKAL_P]          = 0.1037

[EXP2_MARL_JSD_M]         = 0.6499
[EXP2_MARL_JSD_P]         = 0.5206
[EXP2_MARL_DELTA_P]       = +0.0846
[EXP2_MARL_DELTA_M]       = +0.0522
[EXP2_MARL_U]             = 900.00
[EXP2_MARL_P]             = 1.51e-11

[EXP2_CROSS_U]            = 900.00
[E

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
